# Exploración inicial y baseline

## Predicción del riesgo de incumplimiento en Lending Club

**Pregunta predictiva:** ¿Cuál es la probabilidad de que un préstamo personal termine en incumplimiento, utilizando únicamente información disponible al momento de evaluar la solicitud?

**Objetivo:** `Charged Off = 1` y `Fully Paid = 0`.

**Muestra:** 115,000 préstamos, 151 variables, años 2007 a 2018 y plazos de 36 y 60 meses.

## 1. Carga de datos o muestra representativa

Se utiliza una muestra representativa del dataset Lending Club Loan Data. La muestra conserva todas las variables, cubre todo el periodo disponible y contiene resultados definitivos.

In [ ]:
from pathlib import Path
import glob
import warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, confusion_matrix,
                             f1_score, precision_score, recall_score,
                             roc_auc_score)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 160)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

candidates = (glob.glob("/content/lending_club_muestra*.csv.gz")
              + glob.glob("../lending_club_muestra*.csv.gz")
              + glob.glob("lending_club_muestra*.csv.gz"))
if not candidates:
    raise FileNotFoundError("Suba lending_club_muestra.csv.gz a /content/.")

DATA_PATH = Path(candidates[0])
df = pd.read_csv(DATA_PATH, compression="gzip", low_memory=False)
print(f"Archivo: {DATA_PATH}")
print(f"Dimensiones: {df.shape}")

## 2. Número de filas, columnas y tipos de datos

In [ ]:
print(f"Número de filas: {df.shape[0]:,}")
print(f"Número de columnas: {df.shape[1]:,}")

type_summary = (df.dtypes.astype(str).value_counts()
                .rename_axis("tipo").to_frame("cantidad_variables"))
display(type_summary)

variable_types = pd.DataFrame({
    "variable": df.columns,
    "tipo": df.dtypes.astype(str).values,
    "valores_unicos": df.nunique(dropna=True).values,
})
display(variable_types)

## 3. Porcentaje de valores faltantes por variable

In [ ]:
missing_table = pd.DataFrame({
    "variable": df.columns,
    "faltantes": df.isna().sum().values,
    "porcentaje_faltantes": df.isna().mean().mul(100).values,
}).sort_values("porcentaje_faltantes", ascending=False)
display(missing_table)
print("Variables con 50% o más de faltantes:",
      int((missing_table["porcentaje_faltantes"] >= 50).sum()))
print("Variables con 95% o más de faltantes:",
      int((missing_table["porcentaje_faltantes"] >= 95).sum()))

## 4. Distribución de la variable objetivo

La variable objetivo se construye con los resultados definitivos presentes en la muestra.

In [ ]:
df = df[df["loan_status"].isin(["Fully Paid", "Charged Off"])].copy()
df["target"] = (df["loan_status"] == "Charged Off").astype(int)
df["issue_date"] = pd.to_datetime(df["issue_d"], format="%b-%Y", errors="coerce")
df["issue_year"] = df["issue_date"].dt.year

target_distribution = (df["loan_status"].value_counts()
                       .rename_axis("estado").to_frame("cantidad"))
target_distribution["porcentaje"] = target_distribution["cantidad"] / len(df) * 100
display(target_distribution)
print(f"Periodo: {df['issue_date'].min():%Y-%m} a {df['issue_date'].max():%Y-%m}")
print(f"Duplicados completos: {df.duplicated().sum():,}")

## 5. Al menos cinco visualizaciones relevantes

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(15, 17))

sns.countplot(data=df, x="loan_status",
              order=["Fully Paid", "Charged Off"],
              color="#3572A5", ax=axes[0, 0])
axes[0, 0].set_title("1. Distribución de la variable objetivo")
axes[0, 0].set_xlabel("Estado final")
axes[0, 0].set_ylabel("Préstamos")

top_missing = missing_table.head(15).sort_values("porcentaje_faltantes")
axes[0, 1].barh(top_missing["variable"],
                top_missing["porcentaje_faltantes"], color="#C44E52")
axes[0, 1].set_title("2. Variables con más valores faltantes")
axes[0, 1].set_xlabel("Porcentaje")

default_by_year = df.groupby("issue_year")["target"].mean().reset_index()
sns.lineplot(data=default_by_year, x="issue_year",
             y="target", marker="o", color="#C44E52", ax=axes[1, 0])
axes[1, 0].set_title("3. Incumplimiento por año")
axes[1, 0].set_xlabel("Año")
axes[1, 0].set_ylabel("Tasa de incumplimiento")

term_rates = df.groupby("term")["target"].mean().reset_index()
sns.barplot(data=term_rates, x="term", y="target",
            color="#55A868", ax=axes[1, 1])
axes[1, 1].set_title("4. Incumplimiento por plazo")
axes[1, 1].set_xlabel("Plazo")
axes[1, 1].set_ylabel("Tasa de incumplimiento")

purpose_rates = (df.groupby("purpose")["target"].agg(["mean", "size"])
                 .query("size >= 100").sort_values("mean").reset_index())
sns.barplot(data=purpose_rates, y="purpose", x="mean",
            color="#8172B3", ax=axes[2, 0])
axes[2, 0].set_title("5. Incumplimiento por propósito")
axes[2, 0].set_xlabel("Tasa de incumplimiento")
axes[2, 0].set_ylabel("Propósito")

income_limit = df["annual_inc"].quantile(0.99)
income_plot = df[df["annual_inc"].le(income_limit)]
sns.boxplot(data=income_plot, x="loan_status", y="annual_inc",
            order=["Fully Paid", "Charged Off"], showfliers=False,
            color="#4C72B0", ax=axes[2, 1])
axes[2, 1].set_title("6. Ingreso anual por resultado")
axes[2, 1].set_xlabel("Estado final")
axes[2, 1].set_ylabel("Ingreso anual hasta el percentil 99")

plt.tight_layout()
plt.show()
display(default_by_year)
display(term_rates)
display(purpose_rates)

## 6. Identificación de outliers o registros sospechosos

La regla del rango intercuartílico se usa como diagnóstico. Los registros no se eliminan automáticamente porque un valor extremo puede ser válido.

In [ ]:
outlier_variables = ["loan_amnt", "annual_inc", "dti", "delinq_2yrs",
                     "revol_bal", "revol_util", "open_acc", "total_acc"]
rows = []
for column in outlier_variables:
    values = pd.to_numeric(df[column], errors="coerce")
    q1, q3 = values.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    outliers = (values < lower) | (values > upper)
    rows.append({"variable": column, "mínimo": values.min(),
                 "mediana": values.median(), "p99": values.quantile(0.99),
                 "máximo": values.max(),
                 "cantidad_fuera_iqr": int(outliers.sum()),
                 "porcentaje_fuera_iqr": outliers.mean() * 100})
outlier_table = pd.DataFrame(rows).sort_values("porcentaje_fuera_iqr", ascending=False)
display(outlier_table)
print("Ingresos iguales a cero:", int(df["annual_inc"].eq(0).sum()))
print("DTI negativo:", int(df["dti"].lt(0).sum()))
print("Revol util superior a 100:", int(df["revol_util"].gt(100).sum()))

## 7. Discusión inicial de sesgos, leakage y limitaciones

### Leakage

Se excluyen variables generadas después del otorgamiento, como pagos, capital pendiente, recuperaciones, fechas de pago, cobranzas, hardship y settlement. También se excluyen `grade`, `sub_grade`, `int_rate` e `installment` porque reflejan la evaluación y el precio definidos por Lending Club.

### Sesgos y limitaciones

- El dataset contiene préstamos aceptados y no representa a todos los solicitantes.
- La ubicación, vivienda e ingresos pueden actuar como proxies socioeconómicos.
- Las políticas y condiciones económicas cambiaron entre 2007 y 2018.
- `Charged Off` es la clase minoritaria.
- Algunos faltantes son estructurales porque ciertas variables aparecieron después.
- Los resultados definitivos recientes pueden no representar a todos los préstamos emitidos en esos años.
- Los datos corresponden a Lending Club en Estados Unidos.
- El modelo identifica asociaciones históricas, no relaciones causales.

In [ ]:
leakage_columns = [
    "loan_status", "out_prncp", "out_prncp_inv", "total_pymnt",
    "total_pymnt_inv", "total_rec_prncp", "total_rec_int",
    "total_rec_late_fee", "recoveries", "collection_recovery_fee",
    "last_pymnt_d", "last_pymnt_amnt", "next_pymnt_d",
    "last_credit_pull_d", "last_fico_range_high", "last_fico_range_low",
    "grade", "sub_grade", "int_rate", "installment"
]
leakage_review = pd.DataFrame({
    "variable": [c for c in leakage_columns if c in df.columns],
    "decisión": "Excluir del baseline",
})
display(leakage_review)

## 8. Primer baseline simple

Se utiliza regresión logística. La división es estratificada: 70% entrenamiento, 15% validación y 15% prueba.

In [ ]:
numeric_features = ["loan_amnt", "annual_inc", "dti", "delinq_2yrs",
                    "fico_range_low", "fico_range_high", "inq_last_6mths",
                    "open_acc", "pub_rec", "revol_bal", "revol_util", "total_acc"]
categorical_features = ["term", "emp_length", "home_ownership",
                        "verification_status", "purpose", "addr_state",
                        "application_type"]
features = numeric_features + categorical_features
X, y = df[features], df["target"]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE)
X_validation, X_test, y_validation, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=RANDOM_STATE)

display(pd.DataFrame({
    "partición": ["Entrenamiento", "Validación", "Prueba"],
    "filas": [len(X_train), len(X_validation), len(X_test)],
    "tasa_incumplimiento": [y_train.mean(), y_validation.mean(), y_test.mean()],
}))

In [ ]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaler", StandardScaler()),
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=20)),
])
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features),
])
baseline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced",
                                 solver="lbfgs", random_state=RANDOM_STATE)),
])
baseline.fit(X_train, y_train)
print("Baseline entrenado correctamente.")

In [ ]:
def evaluate(model, X_data, y_data, threshold=0.5):
    probabilities = model.predict_proba(X_data)[:, 1]
    predictions = (probabilities >= threshold).astype(int)
    return {
        "filas": len(y_data),
        "tasa_incumplimiento": y_data.mean(),
        "PR-AUC": average_precision_score(y_data, probabilities),
        "ROC-AUC": roc_auc_score(y_data, probabilities),
        "precision": precision_score(y_data, predictions, zero_division=0),
        "recall": recall_score(y_data, predictions),
        "F1": f1_score(y_data, predictions),
        "matriz_confusión": confusion_matrix(y_data, predictions).tolist(),
    }

validation_metrics = evaluate(baseline, X_validation, y_validation)
test_metrics = evaluate(baseline, X_test, y_test)
metrics = pd.DataFrame([validation_metrics, test_metrics],
                       index=["Validación", "Prueba"])
display(metrics.drop(columns="matriz_confusión"))
print("Matriz de validación:", validation_metrics["matriz_confusión"])
print("Matriz de prueba:", test_metrics["matriz_confusión"])
print("Referencia mínima para PR-AUC:", f"{y_test.mean():.4f}")

### Lectura inicial del baseline

PR-AUC es la métrica principal y se compara con la prevalencia de incumplimiento. ROC-AUC mide la capacidad general de ordenamiento. La ponderación balanceada aumenta la detección de incumplimientos, por lo que recall debe interpretarse junto con precisión y la matriz de confusión. No se utiliza accuracy como única medida ni variables posteriores al otorgamiento.